In [1]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# 데이터 로드 
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
submission = pd.read_csv('data/sample_submission.csv')

In [2]:
# 1. 쓸모없는 식별자 및 다중공선성 유발 횟수 컬럼 제거
col_to_drop = [
    'ID', '난자 해동 경과일', 'PGS 시술 여부', 'PGD 시술 여부', '착상 전 유전 검사 사용 여부',
    'IVF 시술 횟수', 'DI 시술 횟수', 'IVF 임신 횟수', 'DI 임신 횟수', 'IVF 출산 횟수', 'DI 출산 횟수'
]
train.drop(columns=[col for col in col_to_drop if col in train.columns], inplace=True)
test.drop(columns=[col for col in col_to_drop if col in test.columns], inplace=True)

# 2. 횟수 컬럼 정수화
def convert_count(val):
    if pd.isna(val): return 0
    return int(str(val).replace('회', '').replace(' 이상', '').strip())

count_cols = [col for col in train.columns if '횟수' in col]
for col in count_cols:
    train[col] = train[col].apply(convert_count)
    test[col] = test[col].apply(convert_count)

# 3. Context 기반 파생 변수 대량 생성 (Feature Engineering)
def create_advanced_features(df):
    # 그룹 1: 시술 효율성 (Process Efficiency)
    df['이식_효율성'] = df['이식된 배아 수'] / (df['총 생성 배아 수'].fillna(0) + 1e-9)
    df['미세주입_효율'] = df['미세주입에서 생성된 배아 수'] / (df['미세주입된 난자 수'].fillna(0) + 1e-9)
    df['난자_생존_효율'] = df['총 생성 배아 수'] / (df['수집된 신선 난자 수'].fillna(0) + 1e-9)
    
    # 그룹 2: 누적 데미지 및 회복력 (Cumulative History)
    df['실패_누적_지수'] = df['총 시술 횟수'] - df['총 임신 횟수']
    df['착상_유지력'] = df['총 출산 횟수'] / (df['총 임신 횟수'].fillna(0) + 1e-9)
    df['클리닉_충성도'] = df['클리닉 내 총 시술 횟수'] / (df['총 시술 횟수'].fillna(0) + 1e-9)
    
    # 그룹 3: 생물학적 골든타임 (Biological Context)
    age_map = {'만18-34세': 30, '만35-37세': 36, '만38-39세': 38.5, '만40-42세': 41, '만43-44세': 43.5, '만45-50세': 47, '알 수 없음': 35}
    df['추정_나이'] = df['시술 당시 나이'].map(age_map)
    df['나이_제곱'] = df['추정_나이'] ** 2  # 나이에 따른 비선형적 성공률 감소 반영
    df['나이_대비_난자보유력'] = df['수집된 신선 난자 수'] / (df['추정_나이'] + 1e-9)
    df['고령_초산_위험군'] = ((df['추정_나이'] >= 35) & (df['총 출산 횟수'] == 0)).astype(int)
    
    # 그룹 4: 불임 원인 복합도 (Cause Complexity)
    inf_cols = [col for col in df.columns if '불임 원인' in col]
    df[inf_cols] = df[inf_cols].fillna(0)
    df['불임_원인_개수'] = df[inf_cols].sum(axis=1)
    df['원인불명_여부'] = (df['불임_원인_개수'] == 0).astype(int)
    
    if '불임 원인 - 남성 요인' in df.columns and '불임 원인 - 여성 요인' in df.columns:
        df['부부_동시_불임'] = ((df['불임 원인 - 남성 요인'] == 1) & (df['불임 원인 - 여성 요인'] == 1)).astype(int)

    return df.drop(columns=['추정_나이']) # 임시 변수 삭제

train = create_advanced_features(train)
test = create_advanced_features(test)

# 4. 인코딩 및 결측치 처리
obj_cols = train.select_dtypes(include=['object']).columns
for col in obj_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train[col].astype(str), test[col].astype(str)]))
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

X_train = train.drop('임신 성공 여부', axis=1).fillna(-1)
y_train = train['임신 성공 여부']
X_test = test.fillna(-1)

In [3]:
# 1. 1차 사전 학습을 통한 피처 중요도(Gain) 추출
pre_model = LGBMClassifier(n_estimators=300, max_depth=5, random_state=42, n_jobs=-1, importance_type='gain')
pre_model.fit(X_train, y_train)

# 중요도가 0이거나 하위 15%인 쓰레기 변수들 찾아내기
importances = pd.Series(pre_model.feature_importances_, index=X_train.columns)
threshold = importances.quantile(0.15)
useless_features = importances[importances <= threshold].index.tolist()

# 검증된 데이터 셋으로 업데이트 (필터링)
X_train_filtered = X_train.drop(columns=useless_features)
X_test_filtered = X_test.drop(columns=useless_features)

print(f"제거된 하위 피처 개수: {len(useless_features)}개")
print(f"최종 학습에 사용될 피처 개수: {X_train_filtered.shape[1]}개\n")

# 2. 필터링된 데이터로 진짜 5-Fold SKF 학습 시작
# 깊이는 4~6 사이인 5로 고정하고, 어제 Optuna에서 얻은 좋은 값들 일부 차용
final_params = {
    'n_estimators': 1500,
    'learning_rate': 0.015,
    'max_depth': 5,            # 요청한 조건 (4~6 사이)
    'num_leaves': 31,          # 2^5 - 1
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': 1.2,   # 타겟 불균형 해소
    'random_state': 42,
    'n_jobs': -1
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros(len(X_test_filtered))
cv_auc_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_filtered, y_train)):
    X_tr, X_val = X_train_filtered.iloc[train_idx], X_train_filtered.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    model = LGBMClassifier(**final_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[early_stopping(stopping_rounds=100, verbose=False)]
    )
    
    val_probs = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, val_probs)
    cv_auc_scores.append(fold_auc)
    
    test_preds += model.predict_proba(X_test_filtered)[:, 1] / skf.n_splits
    
    print(f"Fold {fold+1} AUC: {fold_auc:.4f}")

print("\n" + "="*30)
print(f"최종 평균 CV AUC: {np.mean(cv_auc_scores):.4f}")
print("="*30)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 66228, number of negative: 190123
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.046417 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1500
[LightGBM] [Info] Number of data points in the train set: 256351, number of used features: 65
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.258349 -> initscore=-1.054568
[LightGBM] [Info] Start training from score -1.054568
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [ ]:
# 결과를 sample_submission 형식에 맞게 덮어쓰기
submission['probability'] = test_preds
submission.to_csv('다수context_engineered_submission.csv', index=False)

print("제출 파일(context_engineered_submission.csv) 생성 완료")